In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Fonction Circuit IBM

*Consulte la [référence API](https://docs.quantum.ibm.com/api/functions/ibm-circuit-function)*

> **Note:** * Les fonctions Qiskit sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles de changer.

## Vue d'ensemble
La fonction Circuit IBM&reg; prend des [PUBs abstraits](/guides/primitive-input-output#pubs) comme entrées et retourne des valeurs attendues atténuées comme sorties. Cette fonction de circuit inclut un pipeline automatisé et personnalisé pour permettre aux chercheurs de se concentrer sur la découverte d'algorithmes et d'applications.

## Description
Après avoir soumis ton PUB, tes circuits abstraits et tes observables sont automatiquement transpilés, exécutés sur le matériel et post-traités pour retourner des valeurs attendues atténuées. Pour ce faire, cette fonction combine les outils suivants :

- [Service de transpilation Qiskit](/guides/qiskit-transpiler-service), incluant la sélection automatique des passes de transpilation assistée par IA et heuristiques pour traduire tes circuits abstraits en circuits ISA optimisés pour le matériel
- [Suppression et atténuation d'erreurs requises pour le calcul à l'échelle utilitaire](/guides/error-mitigation-and-suppression-techniques), incluant le twirling de mesure et de porte, le découplage dynamique, l'extinction d'erreurs de lecture par twirling (TREX), l'extrapolation zéro-bruit (ZNE) et l'amplification probabiliste d'erreurs (PEA)
- [Qiskit Runtime Estimator](/guides/get-started-with-estimator), pour exécuter les PUBs ISA sur le matériel et retourner les valeurs attendues atténuées

![Fonction Circuit IBM](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Premiers pas
Authentifie-toi en utilisant ta [clé API](http://quantum.cloud.ibm.com/) et sélectionne la fonction Qiskit comme suit. (Cet extrait suppose que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Exemples
Pour démarrer, essaie cet exemple de base :

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail Qiskit Function ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Les résultats ont le même format qu'un [résultat Estimator](/guides/estimator-input-output) :

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

### Exemples de niveau d'atténuation
L'exemple suivant montre comment définir le niveau d'atténuation :

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


Dans l'exemple suivant, définir le niveau d'atténuation à 1 désactive initialement l'atténuation ZNE, mais définir `zne_mitigation` à `True` remplace la configuration pertinente de `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

### Exemple de sortie
L'extrait de code suivant décrit le format `PrimitiveResult` (et le `PubResult` associé).